# Transcript exploration and evidence mapping

This notebook builds local, human-review artifacts from the source transcripts, the common-question guide, and the manually coded excerpts. Guide wording is a base: candidate question matches are not accepted automatically. Semantic similarity, sentiment, and emotion outputs are not findings. Generated tables contain transcript text and remain ignored by Git.

## Build traceable transcript and NLP review tables

In [ ]:
from pathlib import Path
import importlib
import sys
import pandas as pd
from IPython.display import display
PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / 'scripts').is_dir(): PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))
from scripts._common import ROOT
load_and_validate = importlib.import_module('scripts.01_load_and_validate').load_and_validate
workflow = importlib.import_module('scripts.08_transcript_workflow')
codebook = load_and_validate(export=False)['codes']
review_dir = ROOT / 'outputs' / 'transcript_review'
ALLOW_MODEL_DOWNLOAD = False  # Set True on first use if the embedding model is not cached.
review = workflow.build_transcript_review(codebook, output_dir=review_dir, allow_model_download=ALLOW_MODEL_DOWNLOAD)
counts = pd.DataFrame({'artifact': list(review), 'rows': [len(frame) for frame in review.values()]})
display(counts)

## Audit quote locations

An `exact_normalized` result matches after case, accent, punctuation, and spacing normalization. `candidate_only` is a lexical nearest-window suggestion, never a verified match.

In [ ]:
alignment = review['quote_alignment']
display(alignment.groupby(['alignment_status', 'review_status']).size().rename('excerpt_count').to_frame())
display(alignment.loc[alignment.alignment_status == 'candidate_only', ['excerpt_id', 'participant_id', 'theme', 'transcript_row_start', 'transcript_row_end', 'candidate_similarity', 'codebook_quote', 'candidate_text']].head(20))

## Review adapted question candidates

The three nearest guide prompts are ranked for each interviewer turn. Confirm the question, edit the wording/context note, or reject the candidates. A prompt may be adapted, combined with another, or skipped.

In [ ]:
question_candidates = review['question_alignment_candidates']
question_review = pd.read_csv(review_dir / 'question_alignment_review.csv', encoding='utf-8-sig')
display(question_review[['turn_id', 'participant_id', 'row_start', 'question_like_cue', 'candidate_question_id', 'similarity', 'reviewed_question_id', 'review_decision', 'review_note']].head(30))
# The persisted review file is separate from regenerated candidates.
display(question_candidates[['turn_id', 'interviewer_turn', 'candidate_question_id', 'base_question', 'similarity']].head(12))

## Theme retrieval and affinity map

Theme retrieval surfaces additional transcript passages near existing manual excerpts. Affinity edges are cross-participant semantic neighbors; the editable passage cards are the actual affinity-mapping workspace. Add analyst groups and summaries only after reading each passage in context, and record counterexamples or tensions.

In [ ]:
theme_candidates = review['theme_passage_candidates']
affinity_cards = pd.read_csv(review_dir / 'affinity_passage_review.csv', encoding='utf-8-sig')
theme_review = pd.read_csv(review_dir / 'theme_passage_review.csv', encoding='utf-8-sig')
edge_review = pd.read_csv(review_dir / 'affinity_edge_review.csv', encoding='utf-8-sig')
affinity_edges = review['affinity_candidates']
display(theme_review[['theme_query', 'participant_id', 'passage_id', 'row_start', 'row_end', 'similarity', 'review_label', 'review_note']].head(30))
display(affinity_cards[['passage_id', 'participant_id', 'row_start', 'row_end', 'analyst_affinity_group', 'review_status']].head(30))
display(edge_review[['source_passage_id', 'source_participant_id', 'target_passage_id', 'target_participant_id', 'cosine_similarity', 'accepted_affinity_link', 'analyst_affinity_label', 'review_note']].head(30))

## Optional sentiment and emotion cues

FEEL-IT model weights are not bundled. The models were trained on Italian social-media text and need human review on interview speech. Keep this cell disabled unless model access and study data-handling terms are appropriate. Do not interpret scores as measured emotional states.

In [ ]:
RUN_FEELIT = False
ALLOW_FEELIT_MODEL_DOWNLOAD = False  # Set True only if downloading the model is permitted.
if RUN_FEELIT:
    emotion_cues = workflow.run_feelit_cues(review['transcript_turns'], allow_model_download=ALLOW_FEELIT_MODEL_DOWNLOAD)
    emotion_cues.to_csv(review_dir / 'emotion_cues.csv', index=False, encoding='utf-8-sig')
    display(emotion_cues[['turn_id', 'participant_id', 'sentiment_label', 'sentiment_score', 'emotion_label', 'emotion_score']].head())

## Review protocol

Edit the separate `*_review.csv` tables in a local spreadsheet. Candidate tables are regenerated, while review decisions are carried forward by stable IDs. Preserve model scores and candidate wording beside each decision. See [`docs/transcript_workflow.md`](../docs/transcript_workflow.md) for limits and [`docs/methodology.md`](../docs/methodology.md) for missing study-method details.

## Static visual outputs
The final cells create and display JPEG summaries inline. The affinity network is a navigation aid only; interpret each edge by returning to its transcript excerpt and context.

In [ ]:
visual_module = importlib.import_module("scripts.09_visual_review")
visual_paths = visual_module.build_visuals()
from IPython.display import Image, display
for image_path in visual_paths:
    print(image_path.name)
    display(Image(filename=str(image_path)))